In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from rpy2.robjects import r
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from sklearn.linear_model import LinearRegression
import matplotlib.ticker as mticker

In [ ]:
df_data = pd.read_csv('../../local/output/combined_data_sammed.csv')
df_data.groupby(['cohort', 'label']).size().reset_index(name='count')

In [ ]:
sizes_dict = df_data['cohort'].value_counts().to_dict()
sizes_dict

In [ ]:
# Load cohort classification results
df_cohort_sammed = pd.read_csv('../../local/output/cohort_cls_sammed_results.csv')
df_cohort_sammed.set_index('split', inplace=True)
df_cohort_sammed['model'] = 'SAM-Med3D'

df_cohort_brainiac = pd.read_csv('../../local/output/cohort_cls_brainiac_results.csv')
df_cohort_brainiac.set_index('split', inplace=True)
df_cohort_brainiac['model'] = 'BrainIAC'

df_cohort_dino = pd.read_csv('../../local/output/cohort_cls_dino_results.csv')
df_cohort_dino.set_index('split', inplace=True)
df_cohort_dino['model'] = '3DINO'

df_cohort_dino = pd.read_csv('../../local/output/cohort_cls_dino_results.csv')
df_cohort_dino.set_index('split', inplace=True)
df_cohort_dino['model'] = '3DINO'

df_cohort_medsam2 = pd.read_csv('../../local/output/cohort_cls_medsam2_results.csv')
df_cohort_medsam2.set_index('split', inplace=True)
df_cohort_medsam2['model'] = 'Med-SAM2'

df_cohort_medicalsam2 = pd.read_csv('../../local/output/cohort_cls_medicalsam2_results.csv')
df_cohort_medicalsam2.set_index('split', inplace=True)
df_cohort_medicalsam2['model'] = 'Medical-SAM2'

In [ ]:
df_cohort_sammed.head()

In [ ]:
# Load disease classification results
df_disease_sammed = pd.read_csv('../../local/output/disease_cls_sammed_results.csv')
df_disease_sammed.set_index('split', inplace=True)
df_disease_sammed['model'] = 'SAM-Med3D'
df_disease_sammed['experiment'] = df_disease_sammed['train'] + '_' + df_disease_sammed['test']

df_disease_brainiac = pd.read_csv('../../local/output/disease_cls_brainiac_results.csv')
df_disease_brainiac.set_index('split', inplace=True)
df_disease_brainiac['model'] = 'BrainIAC'
df_disease_brainiac['experiment'] = df_disease_brainiac['train'] + '_' + df_disease_brainiac['test']

df_disease_dino = pd.read_csv('../../local/output/disease_cls_dino_results.csv')
df_disease_dino.set_index('split', inplace=True)
df_disease_dino['model'] = '3DINO'
df_disease_dino['experiment'] = df_disease_dino['train'] + '_' + df_disease_dino['test']

df_disease_medsam2 = pd.read_csv('../../local/output/disease_cls_medsam2_results.csv')
df_disease_medsam2.set_index('split', inplace=True)
df_disease_medsam2['model'] = 'Med-SAM2'
df_disease_medsam2['experiment'] = df_disease_medsam2['train'] + '_' + df_disease_medsam2['test']

df_disease_medicalsam2 = pd.read_csv('../../local/output/disease_cls_medicalsam2_results.csv')
df_disease_medicalsam2.set_index('split', inplace=True)
df_disease_medicalsam2['model'] = 'Medical-SAM2'
df_disease_medicalsam2['experiment'] = df_disease_medicalsam2['train'] + '_' + df_disease_medicalsam2['test']

In [ ]:
df_disease_sammed.head()

In [ ]:
df_cohort_all = pd.concat([df_cohort_sammed, df_cohort_brainiac, df_cohort_dino, df_cohort_medsam2, df_cohort_medicalsam2], axis=0)
df_disease_all = pd.concat([df_disease_sammed, df_disease_brainiac, df_disease_dino, df_disease_medsam2, df_disease_medicalsam2], axis=0)

# remove internal validation results
df_disease_all = df_disease_all[
    df_disease_all["train"] != df_disease_all["test"]
]

In [ ]:
df_disease_all['model'].value_counts()

# Balanced Accuracy

In [ ]:
metric_col = 'balanced_accuracy'

In [ ]:
cohort_summary = (
    df_cohort_all
    .groupby(["experiment", "model"])[metric_col]
    .agg(cohort_mean="mean", cohort_std="std")
    .reset_index()
)
cohort_summary["cohort_pair"] = (
    cohort_summary["experiment"]
    .str.split("_")
    .apply(lambda x: tuple(sorted(x)))
)


In [ ]:
cohort_summary

In [ ]:
disease_summary = (
    df_disease_all
    .groupby(["train", "test", "model"])[metric_col]
    .agg(disease_mean="mean", disease_std="std")
    .reset_index()
)

disease_summary["cohort_pair"] = (
    disease_summary[["train", "test"]]
    .apply(lambda r: tuple(sorted(r)), axis=1)
)
disease_summary

In [ ]:
def add_cohort_pair(df):
    df = df.copy()
    df["cohort_pair"] = (
        df["experiment"]
        .str.split("_")
        .apply(lambda x: "_".join(sorted(x)))
    )
    return df

In [ ]:
cohort_summary["cohort_pair"] = (
    cohort_summary["experiment"]
    .str.split("_")
    .apply(lambda x: tuple(sorted(x)))
)
disease_summary["cohort_pair"] = (
    disease_summary[["train", "test"]]
    .apply(lambda r: tuple(sorted(r)), axis=1)
)

In [ ]:
combined_df = (
    disease_summary
    .merge(
        cohort_summary[["cohort_pair", "model", "cohort_mean", "cohort_std"]],
        on=["cohort_pair", "model"],
        how="inner"
    )
)

In [ ]:
combined_df["cohort_pair"] = combined_df["cohort_pair"].apply(lambda x: "_".join(x))

In [ ]:
combined_df['model'].value_counts()

In [ ]:
combined_df['experiment'] = combined_df['train'] + '_' + combined_df['test'] + '-' + combined_df['model']

In [ ]:
combined_df['train_size'] = combined_df['train'].map(sizes_dict)
combined_df['test_size'] = combined_df['test'].map(sizes_dict)

In [ ]:
exp_names = combined_df['experiment'].tolist()
# -------------------------------
# Model markers
# -------------------------------
marker_map = {'BrainIAC': 'o', 'SAM-Med3D': '*', '3DINO': 'D', 'Med-SAM2': 's', 'Medical-SAM2': 'P'}

# -------------------------------
# Base colors per fine-tuning cohort
# -------------------------------
base_colors = {
    "AIBL": "#1f77b4",        # blue
    "BrainLAT": "#ff7f0e",    # orange
    "OASIS3": "#2ca02c",       # green
    "OASIS4": "#f2e76e",       # yellow
    "NigerianBrain": "#d62728",  # red
    "ADNI": "#9467bd"         # purple
}

# -------------------------------
# Generate 4 shades per fine-tuning cohort
# -------------------------------
def lighten_color(color, amount=0.5):
    c = np.array(mcolors.to_rgb(color))
    white = np.array([1,1,1])
    return tuple(c + (white - c) * amount)

shades_per_cohort = {}
for cohort, base in base_colors.items():
    shades = [base]  # original
    for amt in [0.1, 0.3, 0.5, 0.7]:  # lighter shades
        shades.append(mcolors.to_hex(lighten_color(base, amt)))
    shades_per_cohort[cohort] = shades

# -------------------------------
# Assign a color per fine-tuning → testing cohort pair
# -------------------------------
pair_color_map = {}
# Determine all unique testing cohorts per fine-tuning
ft_to_tests = {}
for name in exp_names:
    pair = name.split('-')[0]
    ft = pair.split('_')[0]
    test = pair.split('_')[1]
    if ft not in ft_to_tests:
        ft_to_tests[ft] = []
    if test not in ft_to_tests[ft]:
        ft_to_tests[ft].append(test)

# Map 4 shades to 4 testing cohorts
for ft, tests in ft_to_tests.items():
    for i, test in enumerate(sorted(tests)):
        pair = f"{ft}_{test}"
        pair_color_map[pair] = shades_per_cohort[ft][i]

# -------------------------------
# Prepare plot data
# -------------------------------
plot_data = []
for i, name in enumerate(exp_names):
    pair = name.split('-')[0]
    model = '-'.join(name.split('-')[1:])
    color = pair_color_map[pair]
    marker = marker_map.get(model)
    plot_data.append((pair, mean_cohort_acc[i], mean_adcn_acc[i],
                      std_cohort_acc[i], std_adcn_acc[i], color, marker))

plot_data.sort(key=lambda x: x[0])

# -------------------------------
# Create figure
# -------------------------------
plt.figure(figsize=(17, 10))
ax = plt.gca()

# Without Error bars
for pair, x, y, xerr, yerr, color, marker in plot_data:
    ax.errorbar(x, y, fmt='none', color=color,
                capsize=4, capthick=1.5, elinewidth=1.5, alpha=0.5)

# Scatter points
for pair, x, y, xerr, yerr, color, marker in plot_data:
    ax.plot(x, y, marker=marker, color=color, markersize=9, alpha=0.8,
            linestyle='none')

# # Plot error bars
# for name, x, y, xerr, yerr, color, marker in plot_data:
#     plt.errorbar(x, y, xerr=xerr, yerr=yerr, fmt='none', color=color,
#                  capsize=5, capthick=2, elinewidth=2, alpha=0.6)

# # Plot scatter points
# for name, x, y, xerr, yerr, color, marker in plot_data:
#     plt.plot(x, y, marker=marker, color=color, markersize=10, alpha=0.8,
#              linestyle='none', label=name)

# Linear regression
X = mean_cohort_acc.reshape(-1, 1)
y = mean_adcn_acc
lr_model = LinearRegression().fit(X, y)
x_line = np.linspace(min(mean_cohort_acc), max(mean_cohort_acc), 100).reshape(-1, 1)
y_line = lr_model.predict(x_line)
ax.plot(x_line, y_line, color='black', linestyle='--', linewidth=2.5, alpha=0.8,
        label='_nolegend_')

# Labels
ax.set_xlabel('Feature-level Cohort Difference: Balanced Accuracy of Cohort Classification (Adaptation vs. Test)', fontsize=14, fontweight='bold')
ax.set_ylabel('Generalizability: Balanced Accuracy of Downstream Classification (AD vs. CN)', fontsize=14, fontweight='bold')
ax.set_title('Generalizability vs. Feature-level Cohort Difference', fontsize=16, fontweight='bold')

# Axes styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(1.8)
ax.spines['left'].set_linewidth(1.8)
ax.tick_params(width=1.8, labelsize=12)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

# Grid
ax.grid(True, alpha=0.2, linestyle='-')

# Axis limits
ax.set_xlim(min(mean_cohort_acc)-0.05, max(mean_cohort_acc)+0.05)
ax.set_ylim(min(mean_adcn_acc)-0.05, max(mean_adcn_acc)+0.05)

# -------------------------------
# Marker legend (model type)
# -------------------------------
marker_handles = [
    Line2D([0],[0], marker='o', color='black', linestyle='none', markersize=9, label='BrainIAC'),
    Line2D([0],[0], marker='*', color='black', linestyle='none', markersize=12, label='SAM-Med3D'),
    Line2D([0],[0], marker='D', color='black', linestyle='none', markersize=12, label='3DINO'),
    Line2D([0],[0], marker='s', color='black', linestyle='none', markersize=12, label='Med-SAM2'),
    Line2D([0],[0], marker='P', color='black', linestyle='none', markersize=12, label='Medical-SAM2')
]
legend1 = ax.legend(handles=marker_handles, title='Model', loc='upper left',
                    bbox_to_anchor=(1.02, 1), fontsize=12, title_fontsize=13)
ax.add_artist(legend1)

# -------------------------------
# Legend for cohort pairs (20 lines)
# -------------------------------
legend_handles = []
seen_pairs = set()
for pair, x, y, xerr, yerr, color, marker in plot_data:
    if pair not in seen_pairs:
        
        handle = Line2D([0], [0], marker='o', color='black',
                        markerfacecolor=color, linestyle='none', markersize=9, label=pair.replace('_', '-'))
        legend_handles.append(handle)
        seen_pairs.add(pair)

legend2 = ax.legend(handles=legend_handles, title='Cohort Pairs (Adaptation-Test)',
                    loc='center left', bbox_to_anchor=(1.02, 0.4),
                    fontsize=10, title_fontsize=12)

# Tight layout
plt.tight_layout(rect=[0,0,0.85,1])
plt.show()
